# COSC 419 — Jersey Number Recognition Pipeline
**Group 9 — Clean Notebook**

This notebook runs the full SoccerNet jersey number recognition pipeline with Top-K keyframe selection on Google Colab (or vast.ai).

### How it works
The pipeline has 9 stages. Six run natively in Python. Three (feature generation, pose estimation, STR) must run as standalone scripts because the original code uses `conda run` internally, which doesn't work in Colab. This notebook handles both cases cleanly.

### Run order
1. **Setup** — Clone repo, install dependencies, download models + data
2. **Pipeline stages 1–9** — Each stage saves results to disk; if you restart, completed stages are skipped
3. **K-sweep** — Re-run combine+eval with different K values (seconds, no GPU needed)

### Important: timm version conflict
- Stages 1–6 need `timm==0.4.9` (pose estimation)
- Stage 7 (STR/PARSeq) needs `timm==0.6.13`
- The notebook switches timm at the right point. **Run stages in order.**

---

In [ ]:
REPO_URL    = "https://github.com/Group-9-Cosc-419/Cosc_419-JerseyNumber-Recognition.git"
BRANCH      = "optimized-jersey"
REPO_DIR    = "/content/jersey-number-pipeline"  # vast.ai uses /workspace

DATASET     = "SoccerNet"
PART        = "test"
TOPK_K      = 5
MINI_TEST_TRACKLETS = 0

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
if not os.path.isdir(REPO_DIR):
    !git clone -b $BRANCH $REPO_URL $REPO_DIR
else:
    print(f"{REPO_DIR} already exists, skipping clone.")
%cd $REPO_DIR

Cloning into '/content/jersey-number-pipeline'...
remote: Enumerating objects: 398, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 398 (delta 44), reused 26 (delta 12), pack-reused 305 (from 1)
Receiving objects: 100% (398/398), 85.57 MiB | 31.82 MiB/s, done.
Resolving deltas: 100% (158/158), done.
/content/jersey-number-pipeline


## 2. Install Dependencies
Runs `setup_env.sh` which installs all packages into the system Python.
No micromamba, no conda environments.

In [ ]:
%cd $REPO_DIR
!chmod +x setup_env.sh
!./setup_env.sh

/content/jersey-number-pipeline
  COSC 419 — Environment Setup

[0/7] Installing prerequisites...

[1/7] Detecting environment...
  PyTorch already installed — keeping existing version
  torch=2.10.0+cu128 | CUDA=True

[2/7] PyTorch...
  Skipped (already installed)

[3/7] Installing core dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.9/86.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 10

In [ ]:
# Patch torch.load for PyTorch 2.6+ (old checkpoints need weights_only=False)
!sed -i 's/torch.load(f, map_location=map_location)/torch.load(f, map_location=map_location, weights_only=False)/g' \
    /usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py 2>/dev/null || true
!sed -i 's/torch.load(f, map_location=map_location)/torch.load(f, map_location=map_location, weights_only=False)/g' \
    /usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py 2>/dev/null || true
print("torch.load patched")

torch.load patched


In [ ]:
%cd $REPO_DIR
!chmod +x download_all.sh
!./download_all.sh

/content/jersey-number-pipeline
  COSC 419 — Download All Assets
  Working dir: /content/jersey-number-pipeline

[1/4] Cloning sub-repositories...
  pose/ViTPose — cloning...
Cloning into 'pose/ViTPose'...
remote: Enumerating objects: 1868, done.
remote: Counting objects: 100% (1207/1207), done.
remote: Compressing objects: 100% (419/419), done.
remote: Total 1868 (delta 861), reused 788 (delta 788), pack-reused 661 (from 2)
Receiving objects: 100% (1868/1868), 10.75 MiB | 28.82 MiB/s, done.
Resolving deltas: 100% (963/963), done.
  sam — cloning...
Cloning into 'sam'...
remote: Enumerating objects: 200, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 200 (delta 85), reused 67 (delta 67), pack-reused 97 (from 1)
Receiving objects: 100% (200/200), 657.08 KiB | 21.20 MiB/s, done.
Resolving deltas: 100% (100/100), done.
  str/parseq — already exists, skipping
  reid/centroids-reid — cloning...
Cloning into 'reid/centroid

In [ ]:
# Unzip train + test sets
import os, shutil

jersey_dir = "data/SoccerNet"
extract_dir = "data/SoccerNet"

for split in ["train", "test"]:
    zip_path = os.path.join(jersey_dir, f"{split}.zip")
    target = os.path.join(extract_dir, split)
    if os.path.isdir(target) and len(os.listdir(target)) > 0:
        print(f"{split} already extracted")
    elif os.path.isfile(zip_path):
        print(f"Unzipping {split}...")
        !unzip -o -q {zip_path} -d {extract_dir}
    else:
        print(f"WARNING: {zip_path} not found")

print(f"Test tracklets: {len(os.listdir('data/SoccerNet/test/images'))}")

Unzipping train...
test already extracted
Test tracklets: 1211


## 4. Apply Patches
These fix compatibility issues with the centroids-reid code and the pipeline's helpers.

In [ ]:
%cd $REPO_DIR

# Patch centroids-reid for pytorch-lightning compatibility
bases_path = "reid/centroids-reid/modelling/bases.py"
with open(bases_path, "r") as f:
    content = f.read()

if "self.hparams = AttributeDict(hparams)" in content:
    content = content.replace("self.hparams = AttributeDict(hparams)", "")
    content = content.replace("self.save_hyperparameters(self.hparams)", "self.save_hyperparameters(hparams)")
    with open(bases_path, "w") as f:
        f.write(content)
    print("Patched centroids-reid/modelling/bases.py")
else:
    print("Already patched")

/content/jersey-number-pipeline
Patched centroids-reid/modelling/bases.py


In [ ]:
%cd $REPO_DIR

# Fix find_best_prediction to filter Top-K BEFORE voting (not after)
import re

with open("helpers.py", "r") as f:
    content = f.read()

# Check if already patched
if "# If topk > 0, keep only the top-K highest confidence" in content:
    print("helpers.py already patched")
else:
    old_func = '''def find_best_prediction(results, useBias=False, topk=1):
    if FILTER_THRESHOLD > 0:
        for entry in results:
            if entry[1] < FILTER_THRESHOLD:
                entry[1] = 0
    unique_predictions = np.unique(results[:, 0])
    weights = []
    #print(unique_predictions)
    weights = []
    for i in range(len(unique_predictions)):
        value = unique_predictions[i]
        rows_with_value = results[np.where(results[:,0]==value)]
        b = get_bias(value) if useBias else 1
        adjusted_prob = rows_with_value[:, 1] * b
        sum_weights = np.sum(adjusted_prob)
        weights.append(sum_weights)

    weights = np.array(weights)
    best_weight = np.max(weights)
    index_of_best = np.argmax(weights)
    best_prediction = unique_predictions[index_of_best] if best_weight > SUM_THRESHOLD else -1

    # Top-K: get indices of top K weights sorted descending
    topk_indices = np.argsort(weights)[::-1][:topk]
    topk_predictions = [int(unique_predictions[i]) for i in topk_indices]

    return best_prediction, unique_predictions, weights, topk_predictions'''

    new_func = '''def find_best_prediction(results, useBias=False, topk=0):
    # If topk > 0, keep only the top-K highest confidence predictions
    # This filters BEFORE voting, so noisy low-confidence predictions are excluded
    if topk > 0 and len(results) > topk:
        sorted_indices = np.argsort(results[:, 1])[::-1]
        results = results[sorted_indices[:topk]]

    if FILTER_THRESHOLD > 0:
        for entry in results:
            if entry[1] < FILTER_THRESHOLD:
                entry[1] = 0
    unique_predictions = np.unique(results[:, 0])
    weights = []
    for i in range(len(unique_predictions)):
        value = unique_predictions[i]
        rows_with_value = results[np.where(results[:,0]==value)]
        b = get_bias(value) if useBias else 1
        adjusted_prob = rows_with_value[:, 1] * b
        sum_weights = np.sum(adjusted_prob)
        weights.append(sum_weights)

    weights = np.array(weights)
    best_weight = np.max(weights)
    index_of_best = np.argmax(weights)
    best_prediction = unique_predictions[index_of_best] if best_weight > SUM_THRESHOLD else -1

    topk_indices = np.argsort(weights)[::-1][:max(topk, 1)]
    topk_predictions = [int(unique_predictions[i]) for i in topk_indices]

    return best_prediction, unique_predictions, weights, topk_predictions'''

    if old_func in content:
        content = content.replace(old_func, new_func)
        with open("helpers.py", "w") as f:
            f.write(content)
        print("Patched find_best_prediction: Top-K now filters BEFORE voting")
    else:
        print("WARNING: Could not find original function to patch. Check helpers.py manually.")

/content/jersey-number-pipeline


In [ ]:
%cd $REPO_DIR

# Fix: always output single best prediction (not a list)
!sed -i 's/final_results\[tracklet\] = topk_predictions if topk > 1 else str(int(best_prediction))/final_results[tracklet] = str(int(best_prediction))/' helpers.py

# Make Top-K configurable via args (replace hardcoded TOPK_K = 5)
with open('main.py', 'r') as f:
    content = f.read()

content = content.replace(
    'TOPK_K = 6  # or whatever K you want',
    '# Use args.topk for configurable K'
)
content = content.replace(
    'helpers.process_jersey_id_predictions(str_result_file, useBias=True, topk=TOPK_K)',
    'helpers.process_jersey_id_predictions(str_result_file, useBias=True, topk=getattr(args, "topk", 5))'
)

with open('main.py', 'w') as f:
    f.write(content)

print("main.py: Top-K now uses args.topk")

/content/jersey-number-pipeline
main.py: Top-K now uses args.topk


In [ ]:
!pip install "numpy<1.24" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 69.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


### Stage 1: Ball Filter

In [ ]:
%cd $REPO_DIR
import sys, importlib, argparse

sys.path.insert(0, REPO_DIR)
import legibility_classifier as lc; importlib.reload(lc)
import helpers; importlib.reload(helpers)
import main; importlib.reload(main)
args = argparse.Namespace()
args.dataset = DATASET
args.part = "test"
args.topk = TOPK_K

args.pipeline = {
    "soccer_ball_filter": True,
    "feat": False, "filter": False, "legible": False, "legible_eval": False,
    "pose": False, "crops": False, "str": False, "combine": False, "eval": False,
}
main.soccer_net_pipeline(args)

/content/jersey-number-pipeline
Determine soccer ball


100%|██████████| 1211/1211 [00:08<00:00, 142.36it/s]

Found 65 balls, Ball list: ['67', '984', '108', '528', '739', '904', '1101', '109', '375', '758', '1005', '304', '767', '1038', '34', '1206', '865', '1137', '939', '832', '68', '1063', '575', '831', '151', '567', '271', '619', '416', '499', '1167', '378', '1160', '19', '850', '1039', '439', '214', '57', '868', '196', '177', '961', '178', '788', '709', '592', '242', '594', '466', '526', '871', '795', '88', '1077', '646', '352', '717', '320', '682', '513', '1165', '866', '623', '576']
Done determine soccer ball


### Stage 2: Feature Generation (ReID)
Runs as standalone script — takes ~84 min on T4 for full dataset.

In [ ]:
!sed -i 's/self\.hparams = AttributeDict(hparams)/self\._hparams = AttributeDict(hparams)/' \
    /content/jersey-number-pipeline/reid/centroids-reid/modelling/bases.py

In [ ]:
%cd $REPO_DIR
import os

features_dir = "out/SoccerNetResults/test/features"
if os.path.isdir(features_dir) and len(os.listdir(features_dir)) > 100:
    print(f"Features already exist ({len(os.listdir(features_dir))} files), skipping.")
else:
    !PYTHONPATH="$REPO_DIR/reid/centroids-reid:$PYTHONPATH" \
        python3 centroid_reid.py \
        --tracklets_folder data/SoccerNet/test/images \
        --output_folder out/SoccerNetResults/test/features

/content/jersey-number-pipeline
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/migration/migration.py:195: PossibleUserWarning: You have multiple `ModelCheckpoint` callback states in this checkpoint, but we found state keys that would end up colliding with each other after an upgrade, which means we can't differentiate which of your checkpoint callbacks needs which states. At least one of your `ModelCheckpoint` callbacks will not be able to reload the state.
  rank_zero_warn(
Lightning automatically upgraded your loaded checkpoint from v1.1.4 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file reid/centroids-reid/models/market1501_resnet50_256_128_epoch_120.ckpt`
using GPU
100% 1211/1211 [1:14:57<00:00,  3.71s/it]


### Stage 3: Gaussian Outlier Filtering

In [ ]:
%cd $REPO_DIR
import os

filter_file = "out/SoccerNetResults/test/main_subject_gauss_th=3.5_r=3.json"
if os.path.isfile(filter_file):
    print("Filter results already exist, skipping.")
else:
    !python3 gaussian_outliers.py \
        --tracklets_folder data/SoccerNet/test/images \
        --output_folder out/SoccerNetResults/test/features

    # Copy to where legibility expects it
    !cp out/SoccerNetResults/test/features/main_subject_gauss*.json out/SoccerNetResults/test/
    !ls out/SoccerNetResults/test/main_subject_gauss*.json

/content/jersey-number-pipeline
100% 1211/1211 [00:32<00:00, 36.89it/s]
'out/SoccerNetResults/test/main_subject_gauss_th=3.5_r=1.json'
'out/SoccerNetResults/test/main_subject_gauss_th=3.5_r=2.json'
'out/SoccerNetResults/test/main_subject_gauss_th=3.5_r=3.json'


### Stage 4: Legibility Classification

In [7]:
%cd $REPO_DIR
import importlib
import legibility_classifier as lc; importlib.reload(lc)
import helpers; importlib.reload(helpers)
import main; importlib.reload(main)

args.pipeline = {
    "soccer_ball_filter": False, "feat": False, "filter": False,
    "legible": True, "legible_eval": True,
    "pose": False, "crops": False, "str": False, "combine": False, "eval": False,
}
main.soccer_net_pipeline(args)

/content/jersey-number-pipeline
Classifying Legibility:


  0%|          | 0/1146 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth



  0%|          | 0.00/83.3M [00:00<?, ?B/s]
 12%|█▏        | 9.88M/83.3M [00:00<00:00, 103MB/s]
 25%|██▍       | 20.8M/83.3M [00:00<00:00, 109MB/s]
 38%|███▊      | 32.0M/83.3M [00:00<00:00, 113MB/s]
 55%|█████▌    | 45.9M/83.3M [00:00<00:00, 125MB/s]
 70%|███████   | 58.6M/83.3M [00:00<00:00, 128MB/s]
100%|██████████| 83.3M/83.3M [00:00<00:00, 127MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
 94%|█████████▎| 1073/1146 [42:31<02:53,  2.38s/it]


KeyboardInterrupt: 

### Stage 5: Pose Estimation (ViTPose)
Runs as standalone script. The `main.py` pose step generates `pose_input.json` first,
then we run `pose.py` directly.

In [ ]:
%cd $REPO_DIR
import importlib, main; importlib.reload(main)

# Generate pose input JSON
args.pipeline = {
    "soccer_ball_filter": False, "feat": False, "filter": False,
    "legible": False, "legible_eval": False,
    "pose": True,  # generates pose_input.json, then fails silently on conda run
    "crops": False, "str": False, "combine": False, "eval": False,
}
main.soccer_net_pipeline(args)

In [ ]:
%cd $REPO_DIR
import os

pose_config = "pose/ViTPose/configs/body/2d_kpt_sview_rgb_img/topdown_heatmap/coco/ViTPose_huge_coco_256x192.py"
pose_checkpoint = "pose/ViTPose/checkpoints/vitpose-h.pth"

# Ensure the output file for pose results is correctly named for the 'train' part
output_pose_results_file = "out/SoccerNetResults/pose_results.json"

if os.path.exists(output_pose_results_file):
    print(f"Pose results already exist: {output_pose_results_file}, skipping.")
else:
    print("Running pose estimation...")
    !python3 pose.py \
        {pose_config} \
        {pose_checkpoint} \
        --img-root / \
        --json-file out/SoccerNetResults/pose_input_train.json \
        --out-json {output_pose_results_file} \
        --batch-size 32 --save-every 500

    # Check if the output file was created
    if os.path.exists(output_pose_results_file):
        print(f"Pose estimation completed. Output file: {output_pose_results_file}")
        !ls -lh {output_pose_results_file}
    else:
        print(f"Error: Pose results file {output_pose_results_file} was not created.")

### Stage 6: Crop Generation

In [8]:
import os
os.chdir('/content/jersey-number-pipeline')
import importlib, main; importlib.reload(main)
import helpers; importlib.reload(helpers)

class Args:
    part = "test"
    pipeline = {
        "soccer_ball_filter": False, "feat": False, "filter": False,
        "legible": False, "legible_eval": False, "pose": False,
        "crops": True,
        "str": False, "combine": False, "eval": False,
    }

args = Args()
main.soccer_net_pipeline(args)

Generate crops


  1%|          | 933/105166 [00:00<00:21, 4771.35it/s]

skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/517/517_425.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/628/628_301.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/628/628_302.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/628/628_300.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/1168/1168_465.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/1168/1168_466.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/1168/1168_464.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/1168/1168_467.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/156/156_474.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data

  2%|▏         | 2421/105166 [00:00<00:20, 4911.07it/s]

skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/589/589_271.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/589/589_282.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/589/589_238.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/824/824_541.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/824/824_598.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/824/824_279.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/824/824_607.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/824/824_296.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/824/824_293.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerN

...

skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_415.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_412.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_229.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_264.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_273.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_15.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_274.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_413.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_416.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNe

100%|█████████▉| 104838/105166 [01:47<00:00, 573.46it/s]

skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_414.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_219.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_410.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/502/502_356.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/762/762_345.jpg, unreliable points
skipping /content/jersey-number-pipeline/./data/SoccerNet/test/images/762/762_346.jpg, unreliable points


100%|██████████| 105166/105166 [01:48<00:00, 973.54it/s]


skipped 3307 out of 105166
Done generating crops
Organizing crops by tracklet
Done organizing crops by tracklet (829 tracklets)


### Stage 7: STR / PARSeq Recognition
**⚠️ timm version switch happens here.** Pose is done, safe to swap.

In [9]:
# Switch timm for PARSeq
!pip install timm==0.6.13 nltk --quiet
print("timm switched to 0.6.13 for PARSeq")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 14.4 MB/s eta 0:00:00
timm switched to 0.6.13 for PARSeq


In [11]:
!MPLBACKEND=Agg python3 str.py \
    models/parseq_epoch=24-step=2575-val_accuracy=95.6044-val_NED=96.3255.ckpt \
    --data_root=out/SoccerNetResults/crops \
    --batch_size=1 --inference \
    --result_file out/SoccerNetResults/jersey_id_results.json

Additional keyword arguments: {'charset_test': '0123456789'}
  0% 0/101859 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
100% 101859/101859 [24:11<00:00, 70.16it/s]


In [24]:
import shutil, os

# Now run combine+eval
import importlib, helpers, main
importlib.reload(helpers); importlib.reload(main)

# Make sure topk is set
args.topk = 50
args.part='test'
args.pipeline = {
    "soccer_ball_filter": False, "feat": False, "filter": False,
    "legible": False, "legible_eval": False, "pose": False,
    "crops": False, "str": False,
    "combine": True, "eval": True,
}
main.soccer_net_pipeline(args)

Combine predictions with quality filter + TTA recovery
TTA recovery: processing 67 rejected tracklets...
TTA recovery: recovered 8/67 tracklets
Saved final results to ./out/SoccerNetResults/final_results.json
1211 1211
Total number of trackslets: 1211, correct: 1072, accuracy: 88.52188274153592%


In [31]:
!python evaluate.py --pred /content/jersey-number-pipeline/out/SoccerNetResults/final_results.json --gt /content/jersey-number-pipeline/data/SoccerNet/test/test_gt.json | tee evaluation_results.txt

------------------------------
EVALUATION RESULTS
------------------------------
Total Samples (GT):    1211
Predictions Provided:  1211
Correct Predictions:   1072
Missing Predictions:   0
------------------------------
FINAL ACCURACY:        88.52%
------------------------------
